In [4]:
# !pip install torch_geometric
# !pip install tensorflow
# !pip install keras
# !pip install zipfile
# !pip install torch-scatter -f https://pytorch-geometric.com/whl/torch-2.0.0+cu118.html

In [1]:
import os
os.chdir('/home/ec2-user/SageMaker/Restart/src')

In [ ]:
%run main.py --ATTENTION_TYPE "Naive" --want_TOP1 71.0 --path '/home/ec2-user/SageMaker/Restart' --epochs 5 --lr 3e-4 --AA 1 --BB 2 --lambd 0.5 --drop_out 0.1 --scale_one_one 10 --res False --ONLY_SIMI True

2024
LEVEL1    TOP1 53.11%    TOP5 66.80%    TOP10 71.69%    TOP20 76.38%
SandR_Model(
  (S_Model): GATModel(
    (gat1): GATLayer(
      (gat_conv): GATConv(1536, 1024, heads=2)
      (batch_norm): BatchNorm1d(2048, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (activation): ReLU()
    )
    (gat2): GATLayer(
      (gat_conv): GATConv(2048, 1024, heads=3)
      (batch_norm): BatchNorm1d(3072, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (dropout): Dropout(p=0.1, inplace=False)
      (activation): ReLU()
    )
    (linear): Linear(in_features=3072, out_features=256, bias=True)
  )
)
------------------------Epoch: 001-----------------------
Update sim!
batch_num: 000     LOSS: 4264.4705
2024
LEVEL1    TOP1 71.59%    TOP5 80.39%    TOP10 82.56%    TOP20 84.54%
Update sim!
batch_num: 001     LOSS: 3776.5468
2024
LEVEL1    TOP1 72.38%    TOP5 82.81%    TOP10 85.57%    TOP20 88.44%
Update sim!
ba

In [9]:
import pandas as pd
import torch
import boto3
import numpy as np
def ret_type(codes, sep = ":"):
    """
    codes should be a list []
    """
    codes_split = [code.split(sep)[0] for code in codes]
    return(codes_split)


os.chdir('/home/ec2-user/SageMaker/GNN_11_6/src')
path = "2023-11-10 08:30:11"
emb = torch.load(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/rel_emb_1.pth')
name_all = pd.read_csv('https://han-attention.s3.amazonaws.com/input/name_desc/name_desc_all_GPT4.csv')
name_all = name_all['sign_id'].values
Final_emb = pd.DataFrame(emb.detach().numpy())
Final_emb.index = name_all
Final_emb.to_csv(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/rel_emb_naive_drop0.1_1111.csv')

s3 = boto3.client('s3')
with open(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/rel_emb_naive_drop0.1_1111.csv', 'rb') as f:
    s3.upload_fileobj(f, 'han-attention', f'input/phenotype_RA/rel_emb_naive_drop0.1_1111.csv')

index = np.sort(np.concatenate([np.where(np.array(ret_type(name_all)) == "RXNORM")[0], np.where(np.array(ret_type(name_all)) == "PheCode")[0]]))
name_all = name_all[index]

emb = emb[index,:]
Final_emb = pd.DataFrame(emb.detach().numpy())
Final_emb.index = name_all
Final_emb.to_csv(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/drug_side_emb_naive_drop0.1_1111.csv')

s3 = boto3.client('s3')
with open(f'/home/ec2-user/SageMaker/GNN_11_6/output/{path}/drug_side_emb_naive_drop0.1_1111.csv', 'rb') as f:
    s3.upload_fileobj(f, 'han-attention', 'input/drug_side/drug_side_emb_naive_drop0.1_1111.csv')